In [ ]:
import pandas as pd
import warnings
import numpy as np
from sklearn.svm import SVC
from rdkit import Chem
from tqdm import tqdm
import warnings
from data_processing import dataset_process
warnings.filterwarnings("ignore", category=FutureWarning)

In [ ]:
# 定义文件路径
dataset_path = "./SAM_data_PCE_15%_0721.csv"

In [ ]:
dataset = pd.read_csv(dataset_path)

In [ ]:
# 指定所需的特征
features_to_include = ['estate', 'substrate', 'pvk_ions', 'pvk_bandgap'] # 'substrate', 'pvk_ions', 'pvk_bandgap'

# 调用 dataset_process 函数
processed_dataset = dataset_process(dataset_path, features_to_include)

In [ ]:
processed_dataset

In [ ]:
processed_dataset.iloc[:,-11:-1]

In [ ]:
processed_dataset.iloc[:,1:-1]

In [ ]:
#按骨架核心分类
df_copy = processed_dataset.copy()

df_1 = pd.DataFrame(columns=df_copy.columns)
df_2 = pd.DataFrame(columns=df_copy.columns)
df_3 = pd.DataFrame(columns=df_copy.columns)
df_4 = pd.DataFrame(columns=df_copy.columns)
df_5 = pd.DataFrame(columns=df_copy.columns)
df_6 = pd.DataFrame(columns=df_copy.columns)
df_7 = pd.DataFrame(columns=df_copy.columns)

# 链状
for index, row in df_copy.iterrows():
    smiles = row['smiles']
    mol = Chem.MolFromSmiles(smiles)
    if mol is not None:
        ring_info = mol.GetRingInfo()
        num_rings = ring_info.NumRings()
        if num_rings == 0:
            df_1 = pd.concat([df_1, pd.DataFrame([row])], ignore_index=True)
            df_copy.drop(index, inplace=True)

# 萘酰胺
patt_1 = Chem.MolFromSmiles('O=C1NC(=O)C2=CC=CC3=C2C1=CC=C3')
for index, row in df_copy.iterrows():
    smiles = row['smiles']
    mol = Chem.MolFromSmiles(smiles)
    if mol.HasSubstructMatch(patt_1):
        df_2 = pd.concat([df_2, pd.DataFrame([row])], ignore_index=True)
        df_copy.drop(index, inplace=True)

# 三苯胺
patt_2 = Chem.MolFromSmiles('C1=CC=C(C=C1)N(C1=CC=CC=C1)C1=CC=CC=C1')
for index, row in df_copy.iterrows():
    smiles = row['smiles']
    mol = Chem.MolFromSmiles(smiles)
    if mol.HasSubstructMatch(patt_2):
        df_3 = pd.concat([df_3, pd.DataFrame([row])], ignore_index=True)
        df_copy.drop(index, inplace=True)

# 咔唑
patt_3 = Chem.MolFromSmiles('N1C2=C(C=CC=C2)C2=C1C=CC=C2')
for index, row in df_copy.iterrows():
    smiles = row['smiles']
    mol = Chem.MolFromSmiles(smiles)
    if mol.HasSubstructMatch(patt_3):
        df_4 = pd.concat([df_4, pd.DataFrame([row])], ignore_index=True)
        df_copy.drop(index, inplace=True)

# 联噻吩
patt_4 = Chem.MolFromSmiles('S1C=CC=C1C1=CC=CS1')
for index, row in df_copy.iterrows():
    smiles = row['smiles']
    mol = Chem.MolFromSmiles(smiles)
    if mol.HasSubstructMatch(patt_4):
        df_5 = pd.concat([df_5, pd.DataFrame([row])], ignore_index=True)
        df_copy.drop(index, inplace=True)

# 单环
for index, row in df_copy.iterrows():
    smiles = row['smiles']
    mol = Chem.MolFromSmiles(smiles)
    if mol is not None:
        ring_info = mol.GetRingInfo()
        num_rings = ring_info.NumRings()
        if num_rings == 1:
            df_6 = pd.concat([df_6, pd.DataFrame([row])], ignore_index=True)
            df_copy.drop(index, inplace=True)

# 其他
df_7 = df_copy.copy()

In [ ]:
print(df_1.shape)
print(df_2.shape)
print(df_3.shape)
print(df_4.shape)
print(df_5.shape)
print(df_6.shape)
print(df_7.shape)

In [ ]:
sam_gen = "./gen_mol_0621_DM_top_10.csv"

In [ ]:
sam_gen_features = ['estate']


In [ ]:
processed_sam_gen = dataset_process(sam_gen, sam_gen_features)

In [ ]:
processed_sam_gen

In [ ]:
columns_to_fill = processed_dataset.columns[-11:-1]
pvk_substrate_info = pd.DataFrame(0, index=processed_sam_gen.index, columns=columns_to_fill)

In [ ]:
pvk_substrate_info

In [ ]:
sam_gen_input = pd.concat([processed_sam_gen.iloc[:,1:], pvk_substrate_info],axis=1)
sam_gen_input

In [ ]:
# 设置最佳超参数组合
best_params = {
    'C': 50,
    'kernel': 'linear',
    'gamma': 0.0001
}
# 用于存储20次实验中每次新分子的预测结果
new_molecule_predictions_all = []

# 遍历重复实验
for i in tqdm(range(20), desc="Training Progress"):
    dfs = [df_1, df_2, df_3, df_4, df_5, df_6]
    df_test = pd.DataFrame()
    df_train = pd.DataFrame()

    for j, df_sub in enumerate(dfs, start=1):
        if len(df_sub) <= 10:
            test_indices = np.random.choice(df_sub.index, size=1, replace=False)
        elif 10 < len(df_sub) <= 20:
            test_indices = np.random.choice(df_sub.index, size=2, replace=False)
        elif 20 < len(df_sub) <= 30:
            test_indices = np.random.choice(df_sub.index, size=3, replace=False)
        elif 30 < len(df_sub) <= 40:
            test_indices = np.random.choice(df_sub.index, size=4, replace=False)
        elif 40 < len(df_sub) <= 50:
            test_indices = np.random.choice(df_sub.index, size=5, replace=False)
        elif 50 < len(df_sub):
            test_indices = np.random.choice(df_sub.index, size=6, replace=False)
        else:
            test_indices = []

        # 将测试集数据添加到 df_test，保持原始索引号
        df_test = pd.concat([df_test, df_sub.loc[test_indices]], ignore_index=False)

        # 将剩余的数据添加到 df_train，保持原始索引号
        train_indices = df_sub.index.difference(test_indices)
        df_train = pd.concat([df_train, df_sub.loc[train_indices]], ignore_index=False)

    # 分离特征和标签
    df_train_X = df_train.iloc[:, 1:-1].astype(float)
    df_test_X = df_test.iloc[:, 1:-1].astype(float)
    df_train_label = df_train['label'].astype(int)
    df_test_label = df_test['label'].astype(int)

    # 使用最佳超参数组合训练模型
    model = SVC(C=best_params['C'],
                kernel=best_params['kernel'],
                gamma=best_params['gamma'])
    model.fit(df_train_X, df_train_label)

    # 在测试集上进行预测
    predictions = model.predict(df_test_X)
    
    # 对新分子进行预测
    new_molecules_X = sam_gen_input.astype(float)  # 假设 sam_gen_input 包含新分子的特征数据
    new_molecule_predictions = model.predict(new_molecules_X)
    
    # 记录每次实验中对新分子的预测结果
    new_molecule_predictions_all.append(new_molecule_predictions)

# 将每次预测结果转换为 NumPy 数组，计算平均值和标准差
new_molecule_predictions_all = np.array(new_molecule_predictions_all)
mean_predictions = np.mean(new_molecule_predictions_all, axis=0)
std_predictions = np.std(new_molecule_predictions_all, axis=0)

# 创建 DataFrame 来记录平均值和标准差
df_predictions = pd.DataFrame({
    'Mean_Prediction': mean_predictions,
    'Std_Deviation': std_predictions
})


In [ ]:
df_predictions

In [ ]:
sam_gen_pred_efficiency = pd.concat([processed_sam_gen.iloc[:,0:1], df_predictions],axis=1)
sam_gen_pred_efficiency

In [ ]:
sam_gen_pred_efficiency.to_csv("./gen_mol_0621_DM_top_10_pred_efficiency.csv", index=False)